Goal: Predict whether a flight will be delayed more than 15 minutes and  arrival delay after departure
Models: XGBoost

For my second project, I used the XGBoost algorithm to predict arrival delays. I created models  trained on the original imbalanced dataset.
For a fast check, I encoded the whole dataset before splitting. This made the workflow quicker, but it could cause data leakage, because information from the test set might influence the training process. Ideally, encoding should be fitted on the training set and then applied to the test set.

In [ ]:

import pandas as pd
import numpy as np

import sklearn # Machine Learning 
from sklearn.model_selection import train_test_split
import xgboost 
from xgboost import XGBClassifier


from sklearn.compose import ColumnTransformer # Data Preprocessing
from sklearn.preprocessing import OneHotEncoder, StandardScaler,LabelEncoder
import category_encoders as ce

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, recall_score,precision_score
from sklearn.metrics import ConfusionMatrixDisplay

In [11]:
df=pd.read_csv('final.csv')


C:\Users\DELL\AppData\Local\Temp\ipykernel_24840\2756008753.py:1: DtypeWarning: Columns (0: ORIGIN_AIRPORT, 1: DESTINATION_AIRPORT, 2: CITY, 3: STATE) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('final.csv')


In [3]:
info_table=pd.DataFrame({
    'Column':df.columns,
    'Missing Values':df.isnull().sum().values,
    'Filling Factor (%)' : (df.shape[0] - df.isnull().sum().values) / df.shape[0] * 100,
    'Dtype':df.dtypes.values,
    
})
info_table=info_table.sort_values(by='Filling Factor (%)',ascending=True)
info_table

,Column,Missing Values,Filling Factor (%),Dtype
20,TEMP_MIN_DEST,1352834,76.751751,float64
21,PRECIP_DEST,1352834,76.751751,float64
19,TEMP_MAX_DEST,1352834,76.751751,float64
16,TEMP_MAX_ORIGIN,1352818,76.752026,float64
17,TEMP_MIN_ORIGIN,1352818,76.752026,float64
18,PRECIP_ORIGIN,1352818,76.752026,float64
23,STATE,486165,91.645327,str
22,CITY,486165,91.645327,str
10,TAXI_OUT,89047,98.469741,float64
11,WHEELS_OFF,89047,98.469741,float64


In [12]:
categorical_cols = ["ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
for col in categorical_cols:
    df[col] = df[col].astype("category")

In [13]:
df_lr = df.copy()

In [32]:
cat_cols = df_lr.select_dtypes(include=['object', 'category']).columns
print("Categorical columns:", cat_cols)

# Convert all columns to string
for col in cat_cols:
    df_lr[col] = df_lr[col].astype(str)

from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# Fit and transform
df_lr[cat_cols] = encoder.fit_transform(df_lr[cat_cols])

Categorical columns: Index([], dtype='str')


In [16]:
df_lr.info()

<class 'pandas.DataFrame'>
RangeIndex: 5819079 entries, 0 to 5819078
Data columns (total 24 columns):
 #   Column               Dtype  
---  ------               -----  
 0   MONTH                int64  
 1   DAY                  int64  
 2   DAY_OF_WEEK          int64  
 3   AIRLINE              float64
 4   FLIGHT_NUMBER        int64  
 5   ORIGIN_AIRPORT       float64
 6   DESTINATION_AIRPORT  float64
 7   SCHEDULED_DEPARTURE  int64  
 8   DEPARTURE_TIME       float64
 9   DEPARTURE_DELAY      float64
 10  TAXI_OUT             float64
 11  WHEELS_OFF           float64
 12  SCHEDULED_TIME       float64
 13  DISTANCE             int64  
 14  SCHEDULED_ARRIVAL    int64  
 15  DELAY_15             int64  
 16  TEMP_MAX_ORIGIN      float64
 17  TEMP_MIN_ORIGIN      float64
 18  PRECIP_ORIGIN        float64
 19  TEMP_MAX_DEST        float64
 20  TEMP_MIN_DEST        float64
 21  PRECIP_DEST          float64
 22  CITY                 float64
 23  STATE                float64
dtypes: float6

In [ ]:
float_cols = df_lr.select_dtypes(include=['float64']).columns
df_lr[float_cols] = df_lr[float_cols].astype('float32')
int_cols = df_lr.select_dtypes(include=['int64']).columns
df_lr[int_cols] = df_lr[int_cols].astype('int32')

In [19]:
df_lr.info()

<class 'pandas.DataFrame'>
RangeIndex: 5819079 entries, 0 to 5819078
Data columns (total 24 columns):
 #   Column               Dtype  
---  ------               -----  
 0   MONTH                int32  
 1   DAY                  int32  
 2   DAY_OF_WEEK          int32  
 3   AIRLINE              float32
 4   FLIGHT_NUMBER        int32  
 5   ORIGIN_AIRPORT       float32
 6   DESTINATION_AIRPORT  float32
 7   SCHEDULED_DEPARTURE  int32  
 8   DEPARTURE_TIME       float32
 9   DEPARTURE_DELAY      float32
 10  TAXI_OUT             float32
 11  WHEELS_OFF           float32
 12  SCHEDULED_TIME       float32
 13  DISTANCE             int32  
 14  SCHEDULED_ARRIVAL    int32  
 15  DELAY_15             int32  
 16  TEMP_MAX_ORIGIN      float32
 17  TEMP_MIN_ORIGIN      float32
 18  PRECIP_ORIGIN        float32
 19  TEMP_MAX_DEST        float32
 20  TEMP_MIN_DEST        float32
 21  PRECIP_DEST          float32
 22  CITY                 float32
 23  STATE                float32
dtypes: float3

In [21]:
# Features (all columns except target)
X = df_lr.drop('DELAY_15', axis=1)

# Target
y = df_lr['DELAY_15']

In [22]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 5819079 entries, 0 to 5819078
Data columns (total 23 columns):
 #   Column               Dtype  
---  ------               -----  
 0   MONTH                int32  
 1   DAY                  int32  
 2   DAY_OF_WEEK          int32  
 3   AIRLINE              float32
 4   FLIGHT_NUMBER        int32  
 5   ORIGIN_AIRPORT       float32
 6   DESTINATION_AIRPORT  float32
 7   SCHEDULED_DEPARTURE  int32  
 8   DEPARTURE_TIME       float32
 9   DEPARTURE_DELAY      float32
 10  TAXI_OUT             float32
 11  WHEELS_OFF           float32
 12  SCHEDULED_TIME       float32
 13  DISTANCE             int32  
 14  SCHEDULED_ARRIVAL    int32  
 15  TEMP_MAX_ORIGIN      float32
 16  TEMP_MIN_ORIGIN      float32
 17  PRECIP_ORIGIN        float32
 18  TEMP_MAX_DEST        float32
 19  TEMP_MIN_DEST        float32
 20  PRECIP_DEST          float32
 21  CITY                 float32
 22  STATE                float32
dtypes: float32(16), int32(7)
memory usage: 510.

In [23]:
y.info()

<class 'pandas.Series'>
RangeIndex: 5819079 entries, 0 to 5819078
Series name: DELAY_15
Non-Null Count    Dtype
--------------    -----
5819079 non-null  int32
dtypes: int32(1)
memory usage: 22.2 MB


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y
)
xb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    
    eval_metric="logloss"
)
xb.fit(X_train, y_train)

y_pred_xgb = xb.predict(X_test)

print("=== XGBoost ===")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

=== XGBoost ===
Accuracy: 0.9505970015878799
              precision    recall  f1-score   support

           0       0.96      0.98      0.97    951128
           1       0.92      0.80      0.86    212688

    accuracy                           0.95   1163816
   macro avg       0.94      0.89      0.91   1163816
weighted avg       0.95      0.95      0.95   1163816



Post-departure model benefits from knowing departure delays, achieving higher detection of arrival delays. Pre-departure models, without this info, perform differently and less accurately.